In [64]:
from aux import *
import matplotlib.pyplot as plt

df = pd.read_pickle("data_family.pkl")
exog = df['exog'].copy().ffill()
sales_by_family = df['sales_by_family'] 
familias = pd.read_csv("familias.csv")
data_by_family = {
    fam: format_as_year_month(sales_by_family[sales_by_family['family'] == fam].copy())
    for fam in sales_by_family['family'].unique()
}
#for fam in data_by_family:
#    data_by_family[fam], _ = limpiar_outliers_x_agrupacion(data_by_family[fam], 'family', 'sale_amount_MM')
name = '0105'
target_col = 'sale_amount_MM'
ignore_col = [] 
negatives_reg_col = []
feat = feature_selection(data_by_family[name], exog, max_lag=4, min_lag=-3,
                         target_col=target_col, ignore_col=ignore_col,
                         negatives_reg_col=negatives_reg_col)
feat = feat[feat['correlación'] > 0.4].reset_index(drop=True)
feat = clean_focus_correlation(feat, group='variable', focus='correlación')
feat = feat.sort_values(by='correlación', ascending=False).reset_index(drop=True)
selected, resumen = collinearity_analysis(data_by_family[name], exog, feat, target_col)
feat = feat[feat['variable'].isin(selected)]
df_final = construir_dataset_familia(name, data_by_family, exog, feat, target_col)
splits = generar_splits(df_final)

/Users/sebastianulloa/.virtualenvs/py310/lib/python3.10/site-packages/statsmodels/tsa/stattools.py:1179: RuntimeWarning: invalid value encountered in divide
  ret = cvf / (np.std(x) * np.std(y))


In [65]:
familias

,Unnamed: 0,hier_family_cd,hier_family_name
0,0,103,FIERRO/HIERRO/ACERO
1,1,105,OBRA GRUESA
2,2,209,HERRAMIENTAS Y MAQUINARIAS
3,3,415,ILUMINACION Y VENTILADORES
4,4,418,MENAJE
5,5,419,DECORACION
6,6,427,ORGANIZACION
7,7,522,AIRE LIBRE
8,8,104,TABIQUERIA/TECHUMBRE/AISLACION
9,9,206,PLOMERIA / GASFITERIA


In [66]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

def plot_family_sales(data_by_family, familias, names, fig_width=18, fig_height=10):
    rows = len(names)
    fig, axes = plt.subplots(rows, 1, figsize=(fig_width, fig_height), sharex=True,
                             constrained_layout=True) 
    if rows == 1:
        axes = [axes]
    for idx, name in enumerate(names):
        serie = data_by_family[name].sale_amount_MM
        x = serie.index
        y = serie.values
        ax = axes[idx]
        ax.plot(x, y, color='steelblue', linewidth=2)
        family_name = familias.loc[familias.hier_family_cd == int(name), 'hier_family_name'].values[0]
        ax.set_title(f"Evolución de Ventas Mensuales - {family_name}", fontsize=13)
        ax.set_ylabel("Ventas (MM)", fontsize=11)
        ax.grid(True, linestyle='--', alpha=0.5)
        if x.dtype.kind in {'M', 'm'}:
            ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
            ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
            ax.tick_params(axis='x', rotation=45)
    axes[-1].set_xlabel("Fecha", fontsize=12)
    plt.show()
names = [ '0105', '0421','0312']  # lista de códigos de familia que quieres graficar
plot_family_sales(data_by_family, familias, names)

/var/folders/_2/b07hnlxx27j566j7xpz5w05m0000gn/T/ipykernel_67832/2740982820.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
from sklearn.preprocessing import MinMaxScaler
from transformers import AutoTokenizer, AutoModelForCausalLM
import numpy as np
import pandas as pd
import torch
_llm_model = None
_tokenizer = None
def fit_predict_eval_llm_forecaster(training_set, test_set, model_params=None):
    """
    Simula la predicción de series de tiempo usando un LLM como si fuera un modelo de forecasting.
    Transforma la serie temporal a texto, el modelo predice tokens que representan valores futuros.
    """
    global _llm_model, _tokenizer

    default_params = {
        'context_length': 128,
        'prediction_length': 24,
        'model_name': 'gpt2'  # o uno de huggingface como mistralai/Mistral-7B-Instruct-v0.2
    }
    if model_params:
        default_params.update(model_params)
    p = default_params

    scaler_y = MinMaxScaler()
    y_train = scaler_y.fit_transform(training_set[['y']]).flatten()
    y_test = scaler_y.transform(test_set[['y']]).flatten()
    y_all = np.concatenate([y_train, y_test])
    total_required_length = p['context_length'] + p['prediction_length']
    if len(y_all) < total_required_length:
        raise ValueError("No hay suficientes datos para el contexto y predicción requeridos.")

    # Inicializa el modelo y tokenizer
    if _llm_model is None or _tokenizer is None:
        _tokenizer = AutoTokenizer.from_pretrained(p['model_name'])
        _llm_model = AutoModelForCausalLM.from_pretrained(p['model_name'])

    # Codifica la serie como texto (e.g., coma separada)
    context_series = y_all[-total_required_length:-p['prediction_length']]
    context_str = ", ".join([f"{v:.3f}" for v in context_series])
    prompt = f"Given the previous values: [{context_str}], predict the next {p['prediction_length']} values:"

    inputs = _tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        outputs = _llm_model.generate(**inputs, max_new_tokens=50, pad_token_id=_tokenizer.eos_token_id)

    decoded = _tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extraer los números predichos desde el texto (simple parsing, no robusto)
    import re
    predicted_str = decoded.split("predict the next")[1]
    predicted_values = re.findall(r"\d+\.\d+", predicted_str)
    y_pred_scaled = np.array(predicted_values[:p['prediction_length']], dtype=float)

    # Inversa del escalamiento
    y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
    index_pred = test_set.index[:len(y_pred)]
    return _llm_model, pd.Series(y_pred, index=index_pred, name='LLMForecast'), scaler_y

In [68]:
import optuna
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_percentage_error

def optimize_llm_forecaster_cv(df, split, n_trials=10):
    """
    Optimiza la longitud de contexto y predicción de un modelo LLM para series de tiempo usando validación cruzada.
    """
    model_dict = {}

    def objective(trial):
        model_params = {
            'context_length': trial.suggest_int('context_length', 24, 72),  # 2 a 6 años (mensual)
            'prediction_length': trial.suggest_int('prediction_length', 6, 18),  # 6 a 18 meses
            'model_name': 'gpt2'  # puedes probar otros como 'tiiuae/falcon-rw-1b' o 'mistralai/Mistral-7B-Instruct-v0.2'
        }

        mape_scores = []
        best_mape = np.inf

        try:
            for train_index, test_index in split:
                training_set = df.iloc[train_index]
                test_set = df.iloc[test_index]

                total_len = len(training_set) + len(test_set)
                required_len = model_params['context_length'] + model_params['prediction_length']

                if total_len < required_len:
                    raise ValueError("No hay suficientes datos para el contexto y predicción requeridos por el modelo.")

                _, y_pred, _ = fit_predict_eval_llm_forecaster(training_set, test_set, model_params)
                y_true = test_set.loc[y_pred.index, 'y']
                mape = mean_absolute_percentage_error(y_true, y_pred)

                mape_scores.append(mape)
                best_mape = min(best_mape, mape)

            model_dict[trial.number] = {
                'params': model_params,
                'mape_best': best_mape
            }

            return np.mean(mape_scores)

        except Exception as e:
            print(f"[ERROR] LLM Trial {trial.number} → {model_params} | {e}")
            return np.inf

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=n_trials)

    trials_data = [
        (
            trial.number,
            trial.params,
            trial.value,
            model_dict.get(trial.number, {}).get('mape_best', np.inf)
        )
        for trial in study.trials
    ]

    trials_df = pd.DataFrame(
        trials_data, columns=['trial_number', 'params', 'mape_mean', 'mape_best']
    )

    return study.best_params, trials_df, model_dict

In [84]:
trials_df_gpt2

,trial_number,params,mape_mean,mape_best,model
0,0,"{'context_length': 34, 'prediction_length': 11}",0.131528,0.075906,gpt2
1,1,"{'context_length': 29, 'prediction_length': 9}",0.092165,0.073783,gpt2
2,2,"{'context_length': 68, 'prediction_length': 16}",0.109789,0.071466,gpt2
3,3,"{'context_length': 49, 'prediction_length': 9}",0.179114,0.086540,gpt2
4,4,"{'context_length': 42, 'prediction_length': 10}",0.224446,0.093203,gpt2


In [85]:
%%capture
from optimizers import optimize_lstm_cv
from optimizers import optimize_gru_cv
from optimizers import optimize_hw_cv
from optimizers import optimize_silverkite_cv
from optimizers import optimize_prophet_cv
from optimizers import optimize_mlp_cv
n_trials=50
#best_params_lstm, trials_df_lstm, model_dict_lstm = optimize_lstm_cv(df_final.assign(ds=df_final.index), splits, n_trials)
best_params_gpt2, trials_df_gpt2, model_dict_gpt2 = optimize_llm_forecaster_cv(df_final.sort_values('ds').reset_index(drop=True), splits, n_trials=50)
#trials_df_lstm['model'] = 'lstm'
trials_df_gpt2['model'] = 'gpt2'
#best_params_hw, trials_df_hw, model_dict_hw = optimize_hw_cv(df_final, splits, n_trials)
#rials_df_hw['model'] = 'holt_winters'
#best_params_sk, trials_df_sk, model_dict_sk = optimize_silverkite_cv(df_final, splits, n_trials)
#trials_df_sk['model'] = 'silverkite'
#best_params_gru, trials_df_gru, model_dict_gru = optimize_gru_cv(df_final.assign(ds=df_final.index), splits, n_trials)
#trials_df_gru['model'] = 'gru'
#best_params_prophet, trials_df_prophet, model_dict_prophet = optimize_prophet_cv(df_final.reset_index(), splits,n_trials)
#trials_df_prophet['model'] = 'prophet'
#best_params_mlp, trials_df_mlp, model_dict_mlp = optimize_mlp_cv(df_final.assign(ds=df_final.index), splits, n_trials)
#trials_df_mlp['model'] = 'MLP'

In [87]:
import pandas as pd
import pickle
results = {}
results['trials'] = pd.concat([
    trials_df_lstm.assign(model='LSTM'),
    trials_df_gpt2.assign(model='GPT2'),
    trials_df_hw.assign(model='HW'),
    #trials_df_sk.assign(model='Silverkite'),
    trials_df_prophet.assign(model='Prophet'),
    trials_df_mlp.assign(model='MLP'),
   # trials_df_gru.assign(model='GRU'),
    trials_df_lstm.assign(model='LSTM')
], ignore_index=True)

results['features']=feat
with open('results_new'+name+'.pkl', 'wb') as file:
    pickle.dump(results, file)

In [60]:
from sklearn.preprocessing import MinMaxScaler
from llama_cpp import Llama
import numpy as np
import pandas as pd
import re

_llm_gguf_model = None  # Global para evitar recarga

def fit_predict_eval_llm_forecaster_gguf(training_set, test_set, model_params=None):
    """
    Simula la predicción de series de tiempo usando un LLM en formato .gguf.
    Si normalize=True, aplica MinMaxScaler. Si no, usa los valores reales.
    """
    global _llm_gguf_model

    default_params = {
        'context_length': 128,
        'prediction_length': 24,
        'model_path': './models/llama-2-7b.Q4_K_M.gguf',
        'max_tokens': 256,
        'n_ctx': 2048,
        'normalize': False
    }
    if model_params:
        default_params.update(model_params)
    p = default_params

    # Validar tamaño
    if p['prediction_length'] > len(test_set):
        raise ValueError(f"prediction_length ({p['prediction_length']}) > test_set ({len(test_set)}).")

    # Normalizar si se indica
    if p['normalize']:
        scaler_y = MinMaxScaler()
        if training_set[['y']].dropna().empty:
            raise ValueError("training_set['y'] está vacío o solo tiene NaN")

        y_train = scaler_y.fit_transform(training_set[['y']]).flatten()
        y_test = scaler_y.transform(test_set[['y']]).flatten()
        y_all = np.concatenate([y_train, y_test])
    else:
        scaler_y = None
        y_train = training_set['y'].values
        y_test = test_set['y'].values
        y_all = np.concatenate([y_train, y_test])

    total_required_length = p['context_length'] + p['prediction_length']
    if len(y_all) < total_required_length:
        raise ValueError("No hay suficientes datos para contexto + predicción.")

    # Cargar modelo solo una vez
    if _llm_gguf_model is None:
        _llm_gguf_model = Llama(model_path=p['model_path'], n_ctx=p['n_ctx'])

    # Crear prompt con ejemplo opcional
    context_series = y_all[-total_required_length:-p['prediction_length']]
    context_str = ", ".join([f"{v:.3f}" for v in context_series])
    prompt = (
        f"Example:\n"
        f"Input: 0.2, 0.3, 0.4\n"
        f"Output: 0.5, 0.6, 0.7\n\n"
        f"Input: {context_str}\n"
        f"Output:"
    )

    output = _llm_gguf_model(prompt, max_tokens=p['max_tokens'], echo=False)
    generated_text = output['choices'][0]['text'].strip()

    # Extraer números
    predicted_values = re.findall(r"\d+\.\d+", generated_text)
    if not predicted_values:
        raise ValueError("El modelo no generó ningún número reconocible.")

    y_pred_raw = np.array(predicted_values, dtype=float)

    # Limitar al tamaño de test_set
    n_values = min(len(y_pred_raw), len(test_set), p['prediction_length'])

    if scaler_y:
        y_pred = scaler_y.inverse_transform(y_pred_raw[:n_values].reshape(-1, 1)).flatten()
    else:
        y_pred = y_pred_raw[:n_values]

    index_pred = test_set.index[:n_values]

    return _llm_gguf_model, pd.Series(y_pred, index=index_pred, name='LLMForecast_GGUF'), scaler_y

ggml_metal_free: deallocating
ggml_metal_mem_pool_free: freeing memory pool, num heaps = 0 (total = 0)
ggml_metal_mem_pool_free: freeing memory pool, num heaps = 0 (total = 0)
ggml_metal_mem_pool_free: freeing memory pool, num heaps = 0 (total = 0)
ggml_metal_mem_pool_free: freeing memory pool, num heaps = 0 (total = 0)
ggml_metal_mem_pool_free: freeing memory pool, num heaps = 0 (total = 0)
ggml_metal_mem_pool_free: freeing memory pool, num heaps = 0 (total = 0)
ggml_metal_mem_pool_free: freeing memory pool, num heaps = 0 (total = 0)
ggml_metal_mem_pool_free: freeing memory pool, num heaps = 0 (total = 0)


In [ ]:
import optuna
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_percentage_error

def optimize_llm_forecaster_cv_gguf(df, split, n_trials=10):
    """
    Optimiza hiperparámetros para el LLM forecaster en .gguf con validación cruzada y normalización opcional.
    """
    model_dict = {}

    def objective(trial):
        # Ajuste dinámico al tamaño mínimo de test_set
        min_test_len = min(len(df.iloc[test_index]) for _, test_index in split)

        model_params = {
            'context_length': trial.suggest_int('context_length', 24, 72),
            'prediction_length': trial.suggest_int('prediction_length', 6, min(18, min_test_len)),
            'model_path': './models/llama-2-7b.Q4_K_M.gguf',
            'max_tokens': trial.suggest_int('max_tokens', 128, 512),
            'n_ctx': 2048,
            'normalize': trial.suggest_categorical('normalize', [True, False])
        }

        mape_scores = []
        best_mape = np.inf

        try:
            for train_index, test_index in split:
                training_set = df.iloc[train_index]
                test_set = df.iloc[test_index]

                total_len = len(training_set) + len(test_set)
                required_len = model_params['context_length'] + model_params['prediction_length']

                if total_len < required_len:
                    raise ValueError("No hay suficientes datos para el contexto y predicción requeridos.")

                _, y_pred, _ = fit_predict_eval_llm_forecaster_gguf(training_set, test_set, model_params)

                y_true = test_set.loc[y_pred.index, 'y']
                mape = mean_absolute_percentage_error(y_true, y_pred)

                mape_scores.append(mape)
                best_mape = min(best_mape, mape)

            model_dict[trial.number] = {
                'params': model_params,
                'mape_best': best_mape
            }

            return np.mean(mape_scores)

        except Exception as e:
            print(f"[ERROR] Trial {trial.number} → {model_params} | {e}")
            return np.inf

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=n_trials)

    trials_data = [
        (
            trial.number,
            trial.params,
            trial.value,
            model_dict.get(trial.number, {}).get('mape_best', np.inf)
        )
        for trial in study.trials
    ]

    trials_df = pd.DataFrame(
        trials_data, columns=['trial_number', 'params', 'mape_mean', 'mape_best']
    )

    return study.best_params, trials_df, model_dict

In [62]:
best_params_gguf, trials_df_gguf, model_dict_gguf = optimize_llm_forecaster_cv_gguf(
    df_final.sort_values('ds').reset_index(drop=True),
    splits,
    n_trials=5
)

llama_model_load_from_file_impl: using device Metal (Apple M1 Max) - 21665 MiB free
llama_model_loader: loaded meta data with 19 key-value pairs and 291 tensors from ./models/llama-2-7b.Q4_K_M.gguf (version GGUF V2)
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = LLaMA v2
llama_model_loader: - kv   2:                       llama.context_length u32              = 4096
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 11008
llama_model_loader: - kv   6:                 llama.rope.dimension_count u32              = 128
llam

KeyboardInterrupt: 

In [19]:
trials_df_gpt2['model'] = 'lagllama'

In [59]:
trials_df_gguf

,trial_number,params,mape_mean,mape_best
0,0,"{'context_length': 72, 'prediction_length': 12...",inf,inf
1,1,"{'context_length': 64, 'prediction_length': 12...",inf,inf
2,2,"{'context_length': 45, 'prediction_length': 12...",inf,inf
3,3,"{'context_length': 43, 'prediction_length': 12...",inf,inf
4,4,"{'context_length': 31, 'prediction_length': 12...",inf,inf
